# Morning class 27/08 — Worksheet 13 SOLUTIONS: recursion   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q6 and Q8 are the two to read carefully. One measures a cost the deck does not
mention; the other is a bug printed on slide 87 in code the deck presents as
correct.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 13 — Recursion. Run this once.
nested = [1, [2, 3, [4, 5]], 6, [[7], 8]]
flat_numbers = [4, 8, 15, 16, 23, 42]
long_list = list(range(1, 101))

print("nested:", nested)
print("flat_numbers:", flat_numbers)

PART A — Factorial, the deck's example

### Question 1

Factorial. -> `1`, `24`, `120`.

Two branches, and both are load-bearing. The **base case** (`n == 1`)
returns without calling anything — it is the only reason this ever stops.
The **recursive case** calls itself with `n - 1`, which is smaller, which is
the only reason it ever reaches the base case.

Drop either one and you get Q11.

`factorial(5)` does not compute anything until it reaches `factorial(1)`.
All five calls are opened first, then the multiplications happen on the way
back out — which is what Q2 and Q4 make visible.

In [ ]:
def factorial(n):
    if n == 1:
        return 1
    else:
        return n * factorial(n - 1)

print(factorial(1))
print(factorial(4))
print(factorial(5))

### Question 2

The trace. -> `called with n = 5, 4, 3, 2, 1`, then the four intermediates `2`, `6`, `24`, `120`, then `120`. **Slide 79's trace starts at `n = 4`.**

Five `called with` lines for `factorial(5)`, counting down. Slide 79 shows
the same code called with 5 and a trace that begins at `n = 4` — while
still ending with `intermediate result for 5 * factorial(4): 120`. Both
cannot be true of one run; the printed trace has lost its first line.

The slide also shows `Final Output: 120` where the code says
`print(factorial(5))`, which prints `120` and nothing else.

The shape is the real lesson. Every `called with` line comes before every
`intermediate result` line: the calls stack up all the way down to the base
case, and only then does anything get multiplied, innermost first. Five
frames were open at once.

In [ ]:
def factorial_traced(n):
    print(f"factorial has been called with n = {n}")
    if n == 1:
        return 1
    else:
        res = n * factorial_traced(n - 1)
        print(f"intermediate result for {n} * factorial({n - 1}): {res}")
        return res

print(factorial_traced(5))

# The slide's output starts at "n = 4" for a call to factorial(5), then ends
# with the n = 5 intermediate line. Both cannot be true of one run: the
# printed trace is missing its first line.

### Question 3

The deck's base case, given a zero. -> `n=5 stops at depth 4`; **`n=0 stops at depth 20 -- the cap, not the base case`**. Then `1 1 120` from the fixed version.

From 5, the chain hits `n == 1` after four steps and stops. From 0, it
never hits it at all — 0, then −1, −2, −3, counting away from the base case
forever. Only the cap stopped it, and without the cap that is a
`RecursionError`.

`n == 1` as a base case works only for `n >= 1`. `n <= 1` catches zero and
any negative, and returns 1 — which is correct for `0!` and is at least
safe for negatives.

A base case must be reachable **from every input you allow**, not just from
the one you tested. That is the same question as "does this loop's
condition ever become false", and Q8 is the deck getting it wrong in
print.

In [ ]:
def depth_reached(n, depth=0):
    if depth >= 20:
        return depth          # gave up
    if n == 1:                # the deck's base case
        return depth
    return depth_reached(n - 1, depth + 1)

print("n=5 stops at depth", depth_reached(5))
print("n=0 stops at depth", depth_reached(0), "-- the cap, not the base case")

def factorial_fixed(n):
    if n <= 1:                # catches 0 and 1, and stops negatives
        return 1
    return n * factorial_fixed(n - 1)

print(factorial_fixed(0), factorial_fixed(1), factorial_fixed(5))

### Question 4

The stack unwinding. -> `going down: 3, 2, 1`, then `bottom`, then `coming back up: 1, 2, 3`.

The counts go down and then back up, and the `coming back up` lines are in
**reverse** order. That is the shape of every recursive function.

When `countdown(3)` calls `countdown(2)`, the outer call does not finish —
it pauses, holding its own `n`, and waits. At `bottom` there are four
paused calls stacked up, each with its own local `n` (worksheet 11 Q10's
`locals()`, four times over). They then finish in reverse order as each
inner call returns.

That stack is the cost. Each frame holds a set of locals and a place to
return to, and there is a hard limit on how many you may have — Q10 and
Q11.

Anything written after the recursive call runs on the way *back*. That is
where Q2's multiplication happens, and it is the part people forget exists.

In [ ]:
def countdown(n):
    if n == 0:
        print("bottom")
        return
    print("going down:   ", n)
    countdown(n - 1)
    print("coming back up:", n)

countdown(3)

PART B — Fibonacci, and what it costs

### Question 5

Fibonacci. -> `[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]`.

Two base cases in one test — `n <= 1` covers both `fib(0) = 0` and
`fib(1) = 1` — and **two** recursive calls rather than one.

That second call is the whole difference between this and factorial, and it
changes the cost from linear to exponential. Slide 86's diagram shows why:
the calls form a *tree*, not a chain, and the same subtrees appear over and
over. `fib(4)` computes `fib(2)` twice; `fib(5)` computes it three times.

Q6 counts them.

In [ ]:
def fib(n):
    if n <= 1:
        return n
    else:
        return fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(11)])

### Question 6

The cost. -> `fib(5)` **15 calls**, `fib(10)` **177**, `fib(20)` **21,891**, `fib(25)` **242,785**.

Five more on `n` multiplies the work by roughly eleven. That is exponential
growth, and it is the concrete version of the deck's "inefficient and
slower" on slide 87.

242,785 calls to compute a number you could reach in 25 additions. `fib(35)`
would be around 30 million; `fib(50)` would not finish today.

The reason is duplication, not depth. The tree recomputes `fib(18)`
thousands of times because no call remembers what any other call worked
out. (The fix has a name — memoisation, a dictionary of answers already
computed — and it turns this back into 25 calls. It is not in this course,
but the shape of the problem is worth recognising.)

Returning `(value, calls)` as a tuple is worksheet 09 Q3's trick doing real
work: it lets the count come back out without a global.

In [ ]:
def fib_counted(n):
    if n <= 1:
        return n, 1
    a, calls_a = fib_counted(n - 1)
    b, calls_b = fib_counted(n - 2)
    return a + b, calls_a + calls_b + 1

for n in [5, 10, 20, 25]:
    value, calls = fib_counted(n)
    print(f"fib({n}) = {value}, computed with {calls} calls")

### Question 7

Recursive against iterative. -> same values every time; `fib(25)` takes **242,785 calls versus 25 passes**, and `agree: True` throughout.

Identical answers, four orders of magnitude apart in work.

The loop keeps two numbers and moves them forward — `a, b = b, a + b` is
tuple unpacking doing a simultaneous swap, so no temporary variable is
needed. One frame, `n` passes, and it will happily do `fib(1000)`.

So the deck's summary is right, and worth being precise about: recursion's
advantage is **readability**, when the problem is genuinely recursive. Its
cost is a stack frame per call and, here, a tree of repeated work. Fibonacci
is the standard teaching example of recursion and one of the worst real uses
of it.

Q9 is what a good one looks like.

In [ ]:
def fib_iterative(n):
    a, b = 0, 1
    passes = 0
    for i in range(n):
        a, b = b, a + b
        passes = passes + 1
    return a, passes

for n in [5, 10, 20, 25]:
    rec_value, calls = fib_counted(n)
    it_value, passes = fib_iterative(n)
    print(f"fib({n}) = {rec_value} / {it_value} -- "
          f"{calls} calls vs {passes} passes, agree: {rec_value == it_value}")

PART C — Slide 87, checked

### Question 8

Slide 87's two versions. -> `n=3` both `6`; `n=1` both `1`; **`n=0  recursive=0  iterative=1`**.

The deck prints these side by side as two ways of doing the same thing.
They disagree, and the recursive one is wrong: **0! is 1**, and it returns
0.

Follow it through. `factorial_recursive(0)`: `0 < 0` is false, `0 == 1` is
false, so it goes to the recursive case and computes
`0 * factorial_recursive(-1)`. That inner call hits `n < 0`, returns 1, and
`0 * 1` is 0.

The `n < 0` clause is what hides it. Without it the call would have run
away and raised, and somebody would have noticed. With it, the function
returns a plausible number for an input it was never designed to handle.
Q3's `n <= 1` gets it right.

The iterative version is correct at zero by accident of construction:
`range(1, 1)` is empty, so `product` stays 1 — worksheet 02 Q3's empty range
being useful for once.

Two implementations of one specification that disagree on one input, in
code presented as a worked example. Test the boundaries: zero, one, empty,
negative.

In [ ]:
def factorial_recursive(n):
    if n < 0 or n == 1:
        return 1
    else:
        return n * factorial_recursive(n - 1)

def factorial_iterative(n):
    product = 1
    for i in range(1, n + 1):
        product = product * i
    return product

for n in [3, 1, 0]:
    print(f"n={n}  recursive={factorial_recursive(n)}  "
          f"iterative={factorial_iterative(n)}")

PART D — Where recursion actually wins

### Question 9

Flattening. -> `[1, 2, 3, 4, 5, 6, 7, 8]`, and `nested` unchanged as `[1, [2, 3, [4, 5]], 6, [[7], 8]]`.

**This is what recursion is for.** The data is a tree of unknown depth, and
the function's shape matches the data's shape: for each item, either it is
a value (keep it) or it is a list (do the same thing to it).

Compare with worksheet 03 Q9, which flattened a list of lists with a nested
comprehension. That worked because the depth was exactly two and known in
advance. Here it is 1, 2 and 3 in different branches, and no fixed number
of nested loops can handle it. The only alternative is to maintain your own
stack in a `while` loop — which is precisely what recursion is doing for
you, using Python's.

The depth is the depth of the *nesting*, not the number of items, so the
recursion limit is no threat: a structure 1,000 levels deep is not a thing
you will meet.

And `nested` is unchanged — this is worksheet 11 Q6's copy version, building
a new list rather than mutating the input.

In [ ]:
def flatten(items):
    out = []
    for item in items:
        if isinstance(item, list):
            out.extend(flatten(item))      # a list: go deeper
        else:
            out.append(item)               # a value: keep it
    return out

print(flatten(nested))
print(nested)

### Question 10

Summing recursively. -> `108 108` and `5050 5050`.

Correct, and a bad idea. The base case is the **empty list** rather than a
number, which is the usual shape for recursing over a sequence, and
`not values` is worksheet 01's truthiness again.

But the recursion is as deep as the list is long. 100 items means 100
stacked frames; 10,000 raises `RecursionError`. `sum()` and a `for` loop
use one frame no matter how long the list is.

There is a second cost that is easy to miss: `values[1:]` **copies** the
rest of the list on every call. Summing 100 items copies about 5,000
elements in total.

The rule of thumb: recurse over **structures whose depth is small and
unbounded in shape** — trees, nested dicts, directory listings, JSON. Loop
over **sequences whose length is large**. Q9 is the first kind; this is the
second.

In [ ]:
def total(values):
    if not values:
        return 0
    return values[0] + total(values[1:])

print(total(flat_numbers), sum(flat_numbers))
print(total(long_list), sum(long_list))

# A list of 10,000 would raise RecursionError: maximum recursion depth
# exceeded. Every element costs one stack frame, and Python's default limit
# is about 1,000. `sum()` and a for loop use one frame regardless of length.
#
# There is a second cost hiding here: values[1:] COPIES the rest of the list
# on every call, so this is quadratic in memory as well as linear in depth.

### Question 11

No base case. -> `120` from the good version, then `RecursionError: maximum recursion depth exceeded`.

Slide 69 says a recursive function with no base condition "will continue to
execute indefinitely". Not quite — Python stops it at roughly 1,000 frames
and raises.

That limit is a safety net, not a design choice you can rely on. Every
frame holds the call's locals and a return address; without the cap the
process would grow until it exhausted its memory and was killed, which is
much harder to diagnose than a traceback.

It is the recursive twin of worksheet 04's runaway `while` loop, with one
important difference: **Python provides this guard for you**, and it does
not provide one for `while True`. A missing base case gives you a
traceback in a second; a missing increment gives you a hung kernel.

And it is the same question in both cases — *what makes this stop?* If you
cannot answer it while writing the function, the function is not finished.

In [ ]:
print(factorial_fixed(5))

def no_base(n):
    return n * no_base(n - 1)

# This is SUPPOSED to raise: RecursionError: maximum recursion depth
# exceeded. Slide 69 says "if a base condition is not indicated, then the
# recursive function will continue to execute indefinitely" -- it does not,
# quite. Python stops it at roughly 1,000 frames.
#
# That limit is a safety net, not a design: each frame holds the call's
# locals, and without the cap the process would exhaust its memory. It is
# the recursive equivalent of worksheet 04's guard on a while loop, except
# that Python provides this one for you.
print(no_base(5))